## 🥇 Gold Table — Business-Ready Aggregates

> **Rule of Gold:** Make data easy for dashboards and reports.

**Business Question:**  
*"How much revenue did each store make each day?"*

**Formula:**  
`revenue = quantity × unit_price`

**What we do:**
1. Read clean data from `silver.sales_clean`
2. Group by `sale_date` and `store_id`
3. Sum up the revenue for each group
4. Save to `gold.daily_revenue_by_store`

In [0]:
# ============================================
# GOLD LAYER: Aggregate for business reports
# ============================================

from pyspark.sql.functions import sum as spark_sum , round , col

# Step 1: Read clean silver data
df_gold = spark.table("cyntexa_dev.silver.sales_clean")

# Step 2: Calculate revenue for each row
# revenue = how many items sold × price of one item
df_gold = df_gold.withColumn("revenue" , col("quantity") * col("unit_price"))

# Step 3: Group by date and store, then add up all revenue
df_gold = df_gold.groupBy("sale_date" , "store_id").agg(
    round(spark_sum(col("revenue")) , 2).alias("total_revenue"),
    spark_sum("quantity").alias("total_items_sold")
).orderBy("sale_date", "store_id")

# Step 4: Save as gold table
df_gold.write\
       .mode("overwrite")\
       .saveAsTable("cyntexa_dev.gold.daily_revenue_by_store")

# Show the final business report
print("\n=== Daily Revenue by Store ===")
spark.table("cyntexa_dev.gold.daily_revenue_by_store").show(20, truncate=False)

## 🏆 Task 7: Extend Gold Layer for Inventory Team

&gt; **Goal:** Build a second gold table for a different stakeholder and explain why it lives in gold.

---

### 📦 Why the Inventory Team Needs a Gold Table

The **inventory team** needs to know:
- Which products are selling fast at which stores
- How much stock is moving per week
- Where to restock before items run out

If they compute this **ad hoc** (on their own every time), they will:
- Write different SQL each time → numbers won't match
- Query raw bronze data → see duplicates and wrong prices
- Wait too long → big tables are slow to scan

By putting this in **gold**, we give them one **trusted, fast, ready-to-use** table.

---

### 🏅 Gold Table Design: `gold.inventory_product_summary`

| Column | Meaning |
|--------|---------|
| `store_id` | Which store sold the item |
| `product_name` | Which product |
| `total_units_sold` | Total quantity sold |
| `total_revenue` | Money made from that product |
| `num_transactions` | How many times it was bought |
| `avg_qty_per_sale` | Average items per transaction |

---

### ✅ Why This Belongs in Gold (Not Ad Hoc)

| If Ad Hoc | If Gold Layer |
|-----------|---------------|
| Each analyst writes their own logic | One single source of truth |
| Reads raw bronze → slow & dirty | Reads clean silver → fast & correct |
| No history or version tracking | Managed by engineers, tested & scheduled |
| Different teams get different answers | Everyone sees the same numbers |

&gt; **Bottom line:** Gold tables save time, prevent mistakes, and keep all teams aligned.

In [0]:
%sql
CREATE OR REPLACE TABLE cyntexa_dev.gold.inventory_product_summary AS 
SELECT 
store_id,
product_name,
-- Total units sold (important for restock decisions)
SUM(quantity) AS total_units_sold,
-- Total revenue (helps prioritize high-value products)
ROUND(SUM(quantity * unit_price),2) AS total_revenue,
-- Number of transactions (shows product popularity)
COUNT(*) AS num_transactions,
-- Average quantity per sale (helps set bundle sizes)
ROUND(AVG(quantity) , 2) AS avg_qty_per_sale
FROM cyntexa_dev.silver.sales_clean
GROUP BY store_id , product_name
ORDER BY store_id , total_revenue DESC

In [0]:
%sql
SELECT * FROM cyntexa_dev.gold.inventory_product_summary

%md
## 🔒 Task 8: Design Note — Data Plane vs Control Plane

&gt; **Goal:** Decide which parts of our pipeline run in the customer's data plane vs. Databricks' control plane, and explain what this means for security.

---

### 🏗️ Quick Definitions

| Term | What It Means | Who Owns It |
|------|---------------|-------------|
| **Data Plane** | Where your actual data lives and where Spark jobs run. This is inside **your cloud account** (AWS / Azure / GCP). | **Customer** (you) |
| **Control Plane** | Where Databricks manages notebooks, user login, job scheduling, and the web UI. This lives in **Databricks' cloud**. | **Databricks** |

&gt; **Simple rule:** Data Plane = your data. Control Plane = Databricks' buttons and screens.

---


---

### 🔧 Task-by-Task Breakdown

| Pipeline Step | Runs In | Why |
|---------------|---------|-----|
| Upload CSV to `bronze.sales_raw` | **Data Plane** | Raw files land in your S3/ADLS bucket |
| Clean data → `silver.sales_clean` | **Data Plane** | Spark jobs run on your clusters, processing your data |
| Aggregate → `gold.inventory_product_summary` | **Data Plane** | Computations happen inside your VPC/VNet |
| Schedule job to run daily | **Control Plane** | Databricks scheduler triggers the job |
| View notebook results | **Control Plane** | UI renders output, but data streams from your account |
| Unity Catalog table permissions | **Control Plane** | Databricks manages who can see what |

---

### 🛡️ What This Means for Network / Security Review

| Security Check | What We Must Do | Why It Matters |
|----------------|-----------------|--------------|
| **Data never leaves Data Plane** | Store bronze/silver/gold in **your** cloud storage | Even Databricks engineers cannot see your raw data |
| **VPC / VNet peering** | Connect your network to Databricks with private links | Traffic stays off the public internet |
| **IAM roles & service principals** | Give Databricks **least-privilege** access | Databricks can only read/write what you allow |
| **Encryption at rest** | Turn on S3/ADLS encryption + Delta Lake encryption | If someone steals the disk, they cannot read it |
| **Encryption in transit** | Use TLS 1.2 for all cluster-to-storage traffic | No one can sniff data moving between systems |
| **IP allowlisting** | Only allow Databricks control plane IPs to reach data plane | Blocks fake Databricks requests |
| **Audit logging** | Enable Unity Catalog audit logs + cloud trail logs | Track who accessed what table and when |

---

### ✅ Final Design Decision

> **All data processing (Bronze → Silver → Gold) stays in the Customer Data Plane.**  
> **Only job orchestration, UI, and access control live in the Databricks Control Plane.**

This follows the **"data gravity"** principle — keep heavy data work close to where the data lives. It also passes security reviews because sensitive sales data never touches Databricks-owned infrastructure.

---

### 📝 Summary for a Security Review

> *"Our sales data remains inside our cloud account at all times. Databricks only sends compute instructions and manages user access — it never stores or processes raw data in its own environment."*